# Generation Parameters: temperature, top_p, and top_k

## Table of contents
- [Setup](#setup)
- [What is temperature?](#temperature)
- [temperature in practice — low vs high](#temperature-demo)
- [What is top_p?](#top-p)
- [top_p in practice](#top-p-demo)
- [What is top_k?](#top-k)
- [Scenario guide — which parameter to reach for](#scenario-guide)
- [Incompatibility with thinking mode](#thinking-incompatibility)
- [Best practices](#best-practices)

**Generation parameters** control how Claude samples the next token at each step of text generation.
Getting them right is the difference between a deterministic, reliable pipeline and a creative but unpredictable one.
This notebook isolates each parameter so you can see exactly what it does and when to use it.

## Setup

Ensure `@anthropic-ai/sdk` is installed and `ANTHROPIC_API_KEY` is set in a `.env` file one level up.

In [ ]:
// npm install @anthropic-ai/sdk


In [ ]:
import Anthropic from 'npm:@anthropic-ai/sdk';
import { load } from 'https://deno.land/std@0.220.0/dotenv/mod.ts';

await load({ envPath: '../.env', export: true });

const MODEL_NAME = 'claude-sonnet-4-6';
const client = new Anthropic({ apiKey: Deno.env.get('ANTHROPIC_API_KEY') ?? '' });
console.log('Client ready. Model:', MODEL_NAME);


## What is temperature?

`temperature` rescales the raw probability distribution over every possible next token before sampling.

| temperature | Effect on distribution | Typical output |
|---|---|---|
| `0` | Collapses to the single highest-probability token (greedy) | Deterministic, repetitive |
| `0.1 – 0.4` | Sharpened — top tokens dominate | Consistent, focused |
| `0.7 – 1.0` (default) | Balanced — moderate variety | Natural prose |
| `> 1.0` | Flattened — low-probability tokens get a real chance | Creative, sometimes incoherent |

**Key mental model:** temperature is a knob on the *shape* of the distribution — how peaked or flat it is.
It does **not** limit which tokens can be sampled; even at low temperature, any token is technically reachable.

> **Anthropic default:** `temperature: 1` (unmodified distribution).

## temperature in practice — low vs high

Call the same prompt three times at `temperature: 0` and three times at `temperature: 1`.
Low temperature should produce identical (or near-identical) answers every time.
High temperature should produce varied ones.

In [ ]:
const PROMPT = 'Name one programming language in a single word.';
const RUNS = 3;

async function runTimes(temperature: number, runs: number): Promise<string[]> {
  const results: string[] = [];
  for (let i = 0; i < runs; i++) {
    const res = await client.messages.create({
      model: MODEL_NAME,
      max_tokens: 20,
      temperature,
      messages: [{ role: 'user', content: PROMPT }],
    });
    const text = res.content[0].type === 'text' ? res.content[0].text.trim() : '';
    results.push(text);
  }
  return results;
}

const lowResults  = await runTimes(0, RUNS);
const highResults = await runTimes(1, RUNS);

console.log('temperature: 0  (deterministic)');
lowResults.forEach((r, i) => console.log(`  Run ${i + 1}: ${r}`));

console.log('\ntemperature: 1  (sampled)');
highResults.forEach((r, i) => console.log(`  Run ${i + 1}: ${r}`));

const lowUnique  = new Set(lowResults).size;
const highUnique = new Set(highResults).size;
console.log(`\nUnique answers — low: ${lowUnique}/${RUNS}, high: ${highUnique}/${RUNS}`);


## What is top_p?

`top_p` implements **nucleus sampling**: before a token is drawn, the vocabulary is sorted by probability
(highest first) and tokens are included until their cumulative probability reaches `top_p`.
Only tokens inside that nucleus are eligible to be sampled.

| top_p | Nucleus size | Effect |
|---|---|---|
| `1.0` (default) | All tokens | No filtering — full distribution |
| `0.9` | Tokens summing to top 90% probability | Cuts the long tail of unlikely tokens |
| `0.5` | Tokens summing to top 50% probability | Much smaller pool — more focused |
| `0.1` | Very few top tokens | Near-deterministic |

**Key mental model:** top_p controls the *size* of the candidate pool, not the shape of the distribution.
It is applied **after** any temperature rescaling.

**temperature vs top_p — the core distinction:**

| Parameter | What it changes | Analogy |
|---|---|---|
| `temperature` | Shape of the distribution (peaked vs flat) | How excited the sampler is |
| `top_p` | Size of the candidate pool (how many tokens are eligible) | How many doors are even on the table |

> **Anthropic recommendation:** adjust one or the other, not both — combined effects are hard to predict.
> Exception: deliberately pairing high `temperature` with low `top_p` to get creative-but-grounded output.

## top_p in practice

Run the same prompt at `top_p: 0.1` (tiny nucleus) and `top_p: 1.0` (full vocabulary) multiple times.
A small nucleus should converge to the same word every time; a full vocabulary should vary.

In [ ]:
async function runWithTopP(top_p: number, runs: number): Promise<string[]> {
  const results: string[] = [];
  for (let i = 0; i < runs; i++) {
    const res = await client.messages.create({
      model: MODEL_NAME,
      max_tokens: 20,
      top_p,
      messages: [{ role: 'user', content: PROMPT }],
    });
    const text = res.content[0].type === 'text' ? res.content[0].text.trim() : '';
    results.push(text);
  }
  return results;
}

const narrowResults = await runWithTopP(0.1, RUNS);
const broadResults  = await runWithTopP(1.0, RUNS);

console.log('top_p: 0.1  (narrow nucleus)');
narrowResults.forEach((r, i) => console.log(`  Run ${i + 1}: ${r}`));

console.log('\ntop_p: 1.0  (full vocabulary)');
broadResults.forEach((r, i) => console.log(`  Run ${i + 1}: ${r}`));

console.log(`\nUnique — narrow: ${new Set(narrowResults).size}/${RUNS}, broad: ${new Set(broadResults).size}/${RUNS}`);


## What is top_k?

`top_k` is a simpler alternative to `top_p`: it limits the candidate pool to the top **k** highest-probability
tokens, regardless of their cumulative probability.

| Parameter | Filters by | Adapts to context? |
|---|---|---|
| `top_p` | Cumulative probability threshold | Yes — nucleus size changes per step |
| `top_k` | Fixed count of top tokens | No — always exactly k candidates |

`top_p` is generally preferred in practice because a fixed `top_k` can be too permissive when the
distribution is already peaked (step where one token has 90% probability) and too restrictive when
the distribution is flat (step with many plausible continuations).

The Claude API accepts `top_k` but it is **not compatible with thinking mode** (same as `temperature`
and `top_p`).

## Scenario guide — which parameter to reach for

The exam rarely asks for default values. It asks: *given this use case, which parameter setting produces
the most reliable or appropriate output?*

| Use case | Recommended setting | Reasoning |
|---|---|---|
| JSON extraction / structured output | `temperature: 0` | Deterministic — same schema every call |
| Code generation (correctness matters) | `temperature: 0 – 0.2` | Low variance — prefer the most likely syntax |
| Customer support (natural but consistent) | `temperature: 0.5 – 0.7` | Moderate variety, still on-topic |
| Creative writing / brainstorming | `temperature: 0.9 – 1.0` | High variety encourages diverse ideas |
| Avoid bizarre completions in creative mode | `top_p: 0.9` + high `temperature` | Cuts long tail while keeping variety |
| Extended / Adaptive Thinking enabled | Do **not** set `temperature`, `top_p`, or `top_k` | Incompatible — causes API error |

**Decision rule of thumb:**
1. Start with `temperature` alone.
2. If you see occasional bizarre outputs at high temperature, add `top_p: 0.9` to trim the tail.
3. Never touch both `temperature` and `top_p` without a clear reason — combined effects are hard to debug.

## Incompatibility with thinking mode

When `thinking` is enabled (`type: 'enabled'` or `type: 'auto'`), the API rejects any request that also
sets `temperature`, `top_p`, or `top_k`.

**Why?** The thinking feature relies on a cryptographic `signature` attached to each reasoning block to
verify the chain of thought has not been tampered with in multi-turn conversations. Sampling parameters
would make the reasoning non-reproducible, which conflicts with that integrity guarantee.

The next cell deliberately triggers this error so you can see the exact message.

In [ ]:
// Deliberately combines temperature with thinking to show the API error.
try {
  await (client.messages.create as any)({
    model: MODEL_NAME,
    max_tokens: 4000,
    temperature: 0.7,
    thinking: { type: 'enabled', budget_tokens: 2000 },
    messages: [{ role: 'user', content: 'Hello.' }],
  });
} catch (e) {
  console.log('Error (temperature + thinking):', String(e));
}

// Same error for top_p:
try {
  await (client.messages.create as any)({
    model: MODEL_NAME,
    max_tokens: 4000,
    top_p: 0.9,
    thinking: { type: 'enabled', budget_tokens: 2000 },
    messages: [{ role: 'user', content: 'Hello.' }],
  });
} catch (e) {
  console.log('Error (top_p + thinking):    ', String(e));
}


## Best practices

### Quick-reference

| Parameter | Default | What it controls | Incompatible with |
|---|---|---|---|
| `temperature` | `1` | Shape of probability distribution | thinking mode |
| `top_p` | `1` | Size of nucleus (cumulative prob. threshold) | thinking mode |
| `top_k` | unset | Fixed count of top candidates | thinking mode |

### Rules

1. **Adjust only one at a time.** Pick `temperature` or `top_p`, not both (unless you have a specific reason).
2. **Deterministic pipelines:** `temperature: 0` — same input always yields the same output.
3. **Creative tasks:** `temperature: 0.9 – 1.0`; add `top_p: 0.9` if outputs become incoherent.
4. **Never set these with thinking enabled** — the API returns a 400 error immediately.
5. **Do not memorise defaults** — instead, know which direction to move each parameter for a given goal.